# This is a test for the whole pipeline using a csv file

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import os
from typing import Dict, List, Optional, Literal
from pydantic import BaseModel, Field
from datetime import datetime

load_dotenv(override=True)

class Activity(BaseModel):
    type: Literal["warmup", "piece", "improvisation", "sight_reading"]

    # repertoire metadata
    piece_name: Optional[str] = None
    composer_name: Optional[str] = None
    key: Optional[str] = None
    section: Optional[str] = None
    bars: Optional[str] = None

    # warmup/exercise metadata
    exercise_name: Optional[str] = None
    tempo: Optional[int] = None
    repetitions: Optional[int] = None

    # general metadata
    focus: Optional[str] = None
    notes: Optional[str] = None


class PracticeSession(BaseModel):
    date: Optional[str] = Field(None, description="YYYY-MM-DD or null")
    duration_min: Optional[int] = Field(None, description="Total duration in minutes or null")
    raw_text: str
    activities: List[Activity]

schema_json = PracticeSession.model_json_schema()


/home/zevaz/projects/pianolog-agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
today = datetime.now().strftime("%Y-%m-%d")
PARSER_PROMPT = f"""
You are a deterministic parser for piano practice logs.

Your task:
- Read the input text.
- Extract structured data.
- Return ONLY valid JSON that matches this schema:

{schema_json}

Additional rules:
- If the date is missing, use today's date: {today}.
- "duration_min" must be an integer number of minutes.
  Convert formats like "~45 min", "50m", "1h20" into minutes.
- Split the text into multiple activities when appropriate.
- "type" must be one of: "warmup", "piece", "improvisation", "sight_reading".
- Do NOT invent activities or fields.
- If a field is not present in the text, set it to null.
- Keep all names and text exactly as written (no normalization).
- "raw_text" must contain the full original input.
- If you cannot identify any activity, return "activities": [] and still include raw_text and date.
- If duration cannot be inferred, set duration_min to null (do not guess).

Output rules:
- Return ONLY valid JSON.
- Do NOT wrap in markdown or code fences.
- Do NOT include explanations or comments.
- Output must start with "{{" and end with "}}".
- Before responding, validate internally that your output is valid JSON and matches the schema exactly.
"""


In [ ]:
NORMALIZER_PROMPT = f"""
You are an input normalizer agent for a database that logs piano practice activity.
Your job is to take a structured practice session JSON and normalize it so that all
names, types, keys, bars, sections, and exercises are consistent with the existing
database and follow strict formatting rules.

The input is ALWAYS a JSON object produced by the parser agent: {schema_json}
You MUST return a JSON object with the exact same schema.

Your output MUST follow the exact same schema.

────────────────────────────────────────────────────────
GENERAL NORMALIZATION RULES
────────────────────────────────────────────────────────

1. VALID ACTIVITY TYPES
   Enforce the allowed activity types:
   ["warmup", "piece", "improvisation", "sight_reading"]

2. LOWERCASE FIELDS
   Convert the following fields to lowercase:
   - exercise_name
   - focus
   - notes
   - section
   - bars (except numbers and hyphens)
   - key (before canonicalization)
   - raw_text (keep as-is from input)

   DO NOT lowercase canonical composer names or canonical piece names.

3. MUSICAL KEY NORMALIZATION
   Normalize musical keys to the format:
      "<Letter><#|b> major" or "<Letter><#|b> minor"
   Rules:
   - Capitalize the letter (A–G)
   - Preserve accidentals (#, b)
   - Always include a space before "major" or "minor"
   - If multiple keys are present, separate with ", "
   Examples:
      "dbminor" → "Db minor"
      "c#Major" → "C# major"
      "g, dminor" → "G major, D minor"

4. MULTIPLE VALUES
   If the parser outputs multiple keys, bars, or sections in a single string,
   split them by comma or semicolon and normalize each.

5. DO NOT REMOVE INFORMATION
   If something cannot be normalized safely, keep the original value.

6. DO NOT INVENT INFORMATION
   Never add composers, pieces, keys, or sections that were not present in the input.

────────────────────────────────────────────────────────
CANONICALIZATION RULES (CRITICAL)
────────────────────────────────────────────────────────

You MUST canonicalize composer names and piece names into a single, consistent,
database‑friendly format.

───────────────
A. COMPOSER NAMES
───────────────
Normalize to full, standard names:

- "chopin", "frédéric chopin", "f. chopin" → "Frédéric Chopin"
- "bach", "js bach", "j. s. bach" → "Johann Sebastian Bach"
- "mozart" → "Wolfgang Amadeus Mozart"
- "beethoven" → "Ludwig van Beethoven"

If unsure, output the most widely accepted full name.

───────────────
B. PIECE NAME CANONICALIZATION
───────────────
Normalize all piece names to the format:

   "<Canonical Title>, Op. <Number> No. <Number>"

Rules:
- Capitalize each word properly.
- Always include “Op.” and “No.” with periods.
- Convert shorthand (25/2 → Op. 25 No. 2).
- Remove extra punctuation.
- If the piece has no opus number, use the standard catalog number (e.g., BWV).
- If BWV is unknown, output the cleanest canonical title without guessing.

───────────────
C. CANONICALIZATION EXAMPLES
───────────────

All of these:
- "fantasie impromptu op66"
- "fantaisie impromptu, op. 66"
- "fantasie impromptu"
- "FI op66"
- "fantaisie impromptu, op66"

→ "Fantaisie Impromptu, Op. 66"

All of these:
- "etude op25 no 2"
- "etude op. 25 no. 2"
- "etude 25/2"
- "etude no. 2, op. 25"

→ "Étude, Op. 25 No. 2"

All of these:
- "sinfonia 11"
- "sinfonia no. 11"
- "bach sinfonia 11"

→ "Sinfonia No. 11, BWV 797"
(If BWV is unknown, output "Sinfonia No. 11")

───────────────
D. IF YOU CANNOT CONFIDENTLY IDENTIFY THE PIECE
───────────────
Output the cleanest possible title without hallucinating:
   "Unknown Piece"

────────────────────────────────────────────────────────
REQUIRED SAFETY RULES FOR PIECE ACTIVITIES
────────────────────────────────────────────────────────

If activity.type == "piece":

1. composer_name MUST NOT be empty.
   If missing:
     - Try to infer from piece_name.
     - If piece_name contains "impromptu" or "étude" or "op. 25", set composer_name = "Frédéric Chopin".
     - If piece_name contains "sinfonia", set composer_name = "Johann Sebastian Bach".
     - If still unknown, set composer_name = "Unknown Composer".

2. piece_name MUST NOT be empty.
   If missing:
     - Try to infer from raw_text.
     - If still unknown, set piece_name = "Unknown Piece".

These rules ensure the database never receives null or empty canonical fields.

====================
OUTPUT RULES
====================
- Output ONLY valid JSON.
- No markdown, no code fences, no explanations.
- Output must start with "{" and end with "}".
- The output must match the PracticeSession schema exactly.

"""

In [4]:
import pandas as pd

df = pd.read_csv("test.csv", encoding="utf-8", nrows=5)
print(df)

print("\n--- Row-by-row inspection ---")
for i, row in df.iterrows():
    print(f"\nRow {i}:")
    print("date:", row["date"])
    print("raw_text:", row["raw_text"])


         date                                           raw_text
0  2025-09-10  sep 10 25, 52 min. Warmup: scales d major + d ...
1  2025-09-11  11/09/25 – 45min. Warmup arpegios, d# minor sc...
2  2025-09-12  12 sept 2025, 50 minutes. Warmup: weak fingers...
3  2025-09-13  13-09-25, 48min. Warmup only today: scales c m...
4  2025-09-14  sept 14 25, 55 min. Warmup: d major, f# minor ...

--- Row-by-row inspection ---

Row 0:
date: 2025-09-10
raw_text: sep 10 25, 52 min. Warmup: scales d major + d minor, weak fingrs, mirror scales. Started reading chopin fantasie impromtu op66 in c# minor, hs slow. Also looked at chopin etude op25 no 2 in F major, just sight reading the first page.

Row 1:
date: 2025-09-11
raw_text: 11/09/25 – 45min. Warmup arpegios, d# minor scale. FI op 66 by chopin: bars 1-8 RH only, focus on rythm + evenness. Etude op25 no 2 by chopin: LH patterns, 60bpm. Felt tired today.

Row 2:
date: 2025-09-12
raw_text: 12 sept 2025, 50 minutes. Warmup: weak fingers, mirror fing

In [5]:
from pianolog.db import PianoLogDB

db = PianoLogDB("practice.db")
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
normalizer = OpenAI()


In [6]:
for i, row in df.iterrows():
    
    messages = [{"role": "system", "content": PARSER_PROMPT}] + [{"role": "user", "content": row["raw_text"]}]
    responseGeminy = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=PracticeSession)
    parsedGe = responseGeminy.choices[0].message.parsed

    normalized_input_json = parsedGe.model_dump_json(indent=2)
    normMessages = [{"role": "system", "content": NORMALIZER_PROMPT}] + [{"role": "user", "content": normalized_input_json}]
    resposneNorm = normalizer.beta.chat.completions.parse(model="gpt-4.1-mini", messages=normMessages, response_format=PracticeSession)
    norm = resposneNorm.choices[0].message.parsed
    db.log_session(norm.model_dump())
